# BSM — CORRECTED RUN — ResNet-34

**NOT paper-faithful** (user-requested variant for a clean, non-leaking evaluation).
The paper-faithful notebooks 01-09 are untouched.

| item | value |
|---|---|
| split | group split by original-image identity, stratified by class (70/15/15) |
| train set | 256 train originals **+ 22 augmented variants of each** = 5888 samples |
| val / test | **original images only** (55 / 56), never augmented |
| optimiser | Adam, lr 1e-3, wd 0 |
| schedule | 60 epochs, full-batch gradient accumulation (micro-batch 32, 1 optimiser step/epoch) |
| checkpoint | best validation accuracy |
| expected runtime | ~14 min on a Colab T4 |

Artefacts: `checkpoints_corr/resnet34/resnet34_corr_best.pt`, `results_corr/metrics_resnet34_corrected.json`.


In [1]:
# --- setup: environment, repo, persistent folders -----------------------------------
import os, sys, json, time, random, subprocess, shutil
from pathlib import Path
from collections import Counter
import numpy as np
import torch
from PIL import Image
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/bsm_paper_replication")
REPO = Path("/content/BSM_Project_2")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/holisticNoobda/BSM-Project-2.git", str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

for sub in ["data", "arrays_corr", "checkpoints_corr", "results_corr", "notebooks"]:
    (PROJECT / sub).mkdir(parents=True, exist_ok=True)

CLASSES = ["Ilmenite", "Rutile", "Zircon", "Garnet", "Monazite", "Sillimanite"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
EXT = [".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"]
print("gpu:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


gpu: True Tesla T4


In [2]:
# --- the 367 ORIGINAL images (no augmentation) --------------------------------------
# Source: KaggleHub "prasannavenkatesant/beach-sand-mineral-bsm" (public, no credentials).
ORIG = REPO / "data" / "originals"

def n_originals():
    if not ORIG.exists():
        return 0
    return sum(1 for c in CLASSES for p in (ORIG / c).glob("*") if p.suffix.lower() in EXT)

if n_originals() < 300:
    import kagglehub
    root = Path(kagglehub.dataset_download("prasannavenkatesant/beach-sand-mineral-bsm"))
    src = None
    for cand in root.rglob("Ilmenite"):
        if cand.is_dir():
            src = cand.parent
            break
    if src is None:
        raise RuntimeError("could not locate the BSM class folders in the downloaded dataset")
    ORIG.mkdir(parents=True, exist_ok=True)
    for d in src.iterdir():
        if d.is_dir():
            shutil.copytree(d, ORIG / d.name, dirs_exist_ok=True)

print("originals per class:", {c: sum(1 for p in (ORIG / c).glob("*") if p.suffix.lower() in EXT) for c in CLASSES})
print("total originals:", n_originals())


originals per class: {'Ilmenite': 62, 'Rutile': 66, 'Zircon': 59, 'Garnet': 60, 'Monazite': 60, 'Sillimanite': 60}
total originals: 367


In [3]:
# --- CORRECTED split: group = original-image identity, stratified by class ---------
# Every original file is its own group; augmented variants are produced ONLY from the
# train fold, so no augmented image can ever come from a val/test original.
items = []
for c in CLASSES:
    for p in sorted((ORIG / c).glob("*")):
        if p.suffix.lower() in EXT:
            items.append({"path": str(p), "cls": c, "y": CLASS_TO_IDX[c], "gid": f"{c}__{p.stem}"})
ys = np.array([it["y"] for it in items])
gids = np.array([it["gid"] for it in items])

from sklearn.model_selection import StratifiedShuffleSplit
sss1 = StratifiedShuffleSplit(n_splits=1, train_size=0.70, random_state=SEED)
tr_rel, rest_rel = next(sss1.split(np.zeros(len(items)), ys))
rest_rel = np.asarray(rest_rel)
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
va_rel, te_rel = next(sss2.split(np.zeros(len(rest_rel)), ys[rest_rel]))
tr = np.asarray(tr_rel)
va = rest_rel[np.asarray(va_rel)]
te = rest_rel[np.asarray(te_rel)]

assert not (set(gids[tr]) & set(gids[va])) and not (set(gids[tr]) & set(gids[te])) and not (set(gids[va]) & set(gids[te])), "group leakage"

split = {"seed": SEED, "n_originals": len(items),
         "train_idx": tr.tolist(), "val_idx": va.tolist(), "test_idx": te.tolist(),
         "group_ids": gids.tolist(), "files": [it["path"] for it in items],
         "class_counts": {n: dict(Counter(CLASSES[i] for i in ys[a])) for n, a in [("train", tr), ("val", va), ("test", te)]}}
with open(PROJECT / "arrays_corr" / "split_manifest_corr.json", "w") as f:
    json.dump(split, f, indent=2)

print("split sizes:", {n: len(a) for n, a in [("train", tr), ("val", va), ("test", te)]})
for n, a in [("train", tr), ("val", va), ("test", te)]:
    print(f"  {n:5s} by class:", split["class_counts"][n])
print("group overlap: train/val", len(set(gids[tr]) & set(gids[va])),
      " train/test", len(set(gids[tr]) & set(gids[te])),
      " val/test", len(set(gids[va]) & set(gids[te])))


split sizes: {'train': 256, 'val': 55, 'test': 56}
  train by class: {'Ilmenite': 43, 'Rutile': 46, 'Monazite': 42, 'Garnet': 42, 'Zircon': 41, 'Sillimanite': 42}
  val   by class: {'Sillimanite': 9, 'Garnet': 9, 'Zircon': 9, 'Ilmenite': 9, 'Monazite': 9, 'Rutile': 10}
  test  by class: {'Rutile': 10, 'Ilmenite': 10, 'Zircon': 9, 'Sillimanite': 9, 'Monazite': 9, 'Garnet': 9}
group overlap: train/val 0  train/test 0  val/test 0


In [4]:
# --- CORRECTED arrays: train = originals + 22 augmented variants of each train original
ARR = PROJECT / "arrays_corr"

def load_resized(path):
    with Image.open(path) as im:
        im = im.convert("RGB").resize((224, 224))
        return np.array(im, dtype=np.uint8)

def aug_variants(img_pil, n=22):
    out = []
    img = TF.resize(img_pil, (256, 256))
    for _ in range(n):
        im = img
        if random.random() < 0.5: im = TF.hflip(im)
        if random.random() < 0.5: im = TF.vflip(im)
        im = TF.rotate(im, random.uniform(-15, 15))
        im = TF.center_crop(TF.resize(im, (256, 256)), 224)
        if random.random() < 0.3: im = TF.adjust_brightness(im, random.uniform(0.8, 1.2))
        if random.random() < 0.3: im = TF.adjust_contrast(im, random.uniform(0.8, 1.2))
        if random.random() < 0.3: im = TF.adjust_saturation(im, random.uniform(0.8, 1.2))
        out.append(np.array(im, dtype=np.uint8))
    return out

cache = {k: (ARR / f) for k, f in [("Xtr", "X_train_corr.npy"), ("ytr", "y_train_corr.npy"),
                                   ("Xva", "X_val_corr.npy"), ("yva", "y_val_corr.npy"),
                                   ("Xte", "X_test_corr.npy"), ("yte", "y_test_corr.npy")]}
if all(p.exists() for p in cache.values()):
    Xtr, ytr = np.load(cache["Xtr"]), np.load(cache["ytr"])
    Xva, yva = np.load(cache["Xva"]), np.load(cache["yva"])
    Xte, yte = np.load(cache["Xte"]), np.load(cache["yte"])
    print("loaded cached corrected arrays from Drive")
else:
    train_items = [items[i] for i in tr]
    val_items = [items[i] for i in va]
    test_items = [items[i] for i in te]
    Xtr_list = [load_resized(it["path"]) for it in train_items]
    ytr_list = [it["y"] for it in train_items]
    random.seed(SEED)
    for it in train_items:
        with Image.open(it["path"]) as im:
            for a in aug_variants(im.convert("RGB"), 22):
                Xtr_list.append(a)
                ytr_list.append(it["y"])
    Xtr = np.stack(Xtr_list); ytr = np.array(ytr_list, dtype=np.int64)
    Xva = np.stack([load_resized(it["path"]) for it in val_items])
    yva = np.array([it["y"] for it in val_items], dtype=np.int64)
    Xte = np.stack([load_resized(it["path"]) for it in test_items])
    yte = np.array([it["y"] for it in test_items], dtype=np.int64)
    np.save(cache["Xtr"], Xtr); np.save(cache["ytr"], ytr)
    np.save(cache["Xva"], Xva); np.save(cache["yva"], yva)
    np.save(cache["Xte"], Xte); np.save(cache["yte"], yte)

print("train (256 originals + 22x augmented):", Xtr.shape, Xtr.dtype)
print("val originals:", Xva.shape, "| test originals:", Xte.shape)
print("train class counts:", dict(Counter(ytr.tolist())))


loaded cached corrected arrays from Drive
train (256 originals + 22x augmented): (5888, 224, 224, 3) uint8
val originals: (55, 224, 224, 3) | test originals: (56, 224, 224, 3)
train class counts: {0: 989, 1: 1058, 4: 966, 3: 966, 2: 943, 5: 966}


In [5]:
# --- normalisation (train-fold channel mean/std) + datasets --------------------------
import common

def chan_stats(x_uint8):
    f = x_uint8.astype(np.float32) / 255.0
    return f.mean(axis=(0, 1, 2)).astype(np.float32), f.std(axis=(0, 1, 2)).astype(np.float32)

mean, std = chan_stats(Xtr)
np.save(PROJECT / "arrays_corr" / "norm_mean_corr.npy", mean)
np.save(PROJECT / "arrays_corr" / "norm_std_corr.npy", std)
print("train-fold channel mean:", mean, " std:", std)

train_ds = common.ArrayDataset(Xtr, ytr, mean, std)
val_ds = common.ArrayDataset(Xva, yva, mean, std)
test_ds = common.ArrayDataset(Xte, yte, mean, std)

MICRO = 32
train_loader = DataLoader(train_ds, batch_size=MICRO, shuffle=False, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)
print("micro-batches per epoch:", len(train_loader), "| samples/epoch:", len(train_ds))


train-fold channel mean: [0.05678793 0.05678793 0.05678793]  std: [0.23830219 0.23830219 0.23830219]
micro-batches per epoch: 184 | samples/epoch: 5888


In [6]:
# --- train ResNet-34 (CORRECTED) ------------------------------------------------------
# Same optimisation protocol as the faithful run: 60 epochs, full-batch gradient
# accumulation (micro-batch 32, ONE optimiser step per epoch), Adam lr=1e-3, wd=0,
# best-validation-accuracy checkpoint. Only the data pipeline differs.
from models import resnet34

torch.manual_seed(SEED)
model = resnet34().cuda()
print("ResNet-34 params:", f"{sum(p.numel() for p in model.parameters()):,}")

CKPT = PROJECT / "checkpoints_corr" / "resnet34"
RES = PROJECT / "results_corr"
CKPT.mkdir(parents=True, exist_ok=True)
RES.mkdir(parents=True, exist_ok=True)

t0 = time.time()
hist = common.train_model(
    model, train_loader, val_loader, torch.device("cuda"),
    epochs=60, full_batch=True,
    ckpt_dir=CKPT, history_path=RES / "history_resnet34_corrected.json",
    tag="resnet34_corr", lr=1e-3, resume=True,
)
print(f"TRAINING DONE in {(time.time()-t0)/60:.1f} min")
print("best epoch:", hist.get("best_epoch"), "| best val acc:", round(hist.get("best_val_acc", -1), 4),
      "| final train acc:", round(hist["train_acc"][-1], 4))


ResNet-34 params: 21,287,750


resnet34_corr epoch 01/60  train_loss 1.9064 acc 0.1797  val_loss 4106.3452 acc 0.1636  (26.6s)


resnet34_corr epoch 02/60  train_loss 3.2059 acc 0.1829  val_loss 8.7875 acc 0.3273  (19.4s)


resnet34_corr epoch 03/60  train_loss 2.0454 acc 0.2327  val_loss 11.0792 acc 0.1818  (20.0s)


resnet34_corr epoch 04/60  train_loss 2.5255 acc 0.1462  val_loss 3.0325 acc 0.2182  (19.2s)


...

resnet34_corr epoch 57/60  train_loss 0.9261 acc 0.6793  val_loss 1.7059 acc 0.4364  (19.2s)


resnet34_corr epoch 58/60  train_loss 0.8748 acc 0.6974  val_loss 1.6967 acc 0.3636  (19.9s)


resnet34_corr epoch 59/60  train_loss 0.8207 acc 0.7328  val_loss 1.6168 acc 0.4909  (19.2s)


resnet34_corr epoch 60/60  train_loss 0.7664 acc 0.7724  val_loss 1.3508 acc 0.5455  (19.2s)


TRAINING DONE in 19.7 min
best epoch: 60 | best val acc: 0.5455 | final train acc: 0.7724


In [7]:
# --- evaluate on ORIGINAL test images (clean, never augmented) ----------------------
device = torch.device("cuda")
model = resnet34().to(device)
ck = torch.load(CKPT / "resnet34_corr_best.pt", map_location=device, weights_only=True)
model.load_state_dict(ck)
model.eval()

_, _, logits, y_true = common.evaluate(model, test_loader, device)
y_pred = logits.argmax(1)
acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=range(6))
specs = []
for i in range(6):
    tn = cm.sum() - cm[i, :].sum() - cm[:, i].sum() + cm[i, i]
    fp = cm[:, i].sum() - cm[i, i]
    specs.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
spec = float(np.mean(specs))

hist = json.load(open(RES / "history_resnet34_corrected.json"))
best_epoch = int(np.argmax(hist["val_acc"])) + 1

print(f"\n=== CORRECTED ResNet-34 TEST (originals only, best epoch {best_epoch}) ===")
print(f"accuracy {acc:.4f} | macro precision {prec:.4f} | macro recall {rec:.4f} | macro F1 {f1:.4f} | macro specificity {spec:.4f}")
print("confusion matrix (rows = true):")
print(cm)

results = {
    "model": "resnet34", "type": "CORRECTED", "notebook": "07_corr_resnet34",
    "best_epoch": best_epoch,
    "train_set": "train originals + 22 augmented variants of each train original (group split)",
    "eval_set": "original test images only (group split)",
    "n_train": int(len(train_ds)), "n_val": int(len(val_ds)), "n_test": int(len(y_true)),
    "train_minutes": round((time.time() - t0) / 60, 2),
    "test_split": {"accuracy": float(acc), "macro_precision": float(prec), "macro_recall": float(rec),
                   "macro_f1": float(f1), "macro_specificity": spec,
                   "confusion_matrix": cm.tolist(), "n_test": int(len(y_true))},
}
with open(RES / "metrics_resnet34_corrected.json", "w") as f:
    json.dump(results, f, indent=2)
print("saved", RES / "metrics_resnet34_corrected.json")
print("EVAL DONE")



=== CORRECTED ResNet-34 TEST (originals only, best epoch 60) ===
accuracy 0.4464 | macro precision 0.4722 | macro recall 0.4407 | macro F1 0.4249 | macro specificity 0.8889
confusion matrix (rows = true):
[[5 2 0 0 0 3]
 [2 7 1 0 0 0]
 [1 2 4 0 2 0]
 [0 4 0 1 2 2]
 [0 1 0 1 5 2]
 [2 1 1 0 2 3]]
saved /content/drive/MyDrive/bsm_paper_replication/results_corr/metrics_resnet34_corrected.json
EVAL DONE


In [8]:
# --- persist the executed notebook to the project folder on Drive -------------------
import glob, shutil
os.makedirs(os.path.join(PROJECT, "notebooks"), exist_ok=True)
cands = glob.glob("/content/drive/MyDrive/Colab Notebooks/*.ipynb")
if cands:
    newest = max(cands, key=os.path.getmtime)
    dst = os.path.join(PROJECT, "notebooks", "_autosave_07_corr_resnet34.ipynb")
    shutil.copy(newest, dst)  # staging only; the executed notebook is copied in afterwards
    print("copied", newest, "->", dst)
else:
    print("no autosaved notebook found in Drive/Colab Notebooks")


copied /content/drive/MyDrive/Colab Notebooks/Untitled7.ipynb -> /content/drive/MyDrive/bsm_paper_replication/notebooks/_autosave_07_corr_resnet34.ipynb
